# Agent lab : un assistant de studio, sans puis avec mémoire

Objectif : voir la différence entre un agent **sans mémoire** (chaque question part de zéro)
et un agent **avec mémoire de session** (il se souvient de la conversation).

- SDK : `anthropic` (officiel, client asynchrone `AsyncAnthropic`)
- Modèle : `claude-opus-5-5`, effort `low` (suffisant pour une conversation courte)
- La clé est lue depuis `.env` : elle n'apparaît jamais dans le notebook.

## 1. Préparation

On charge `.env`, on vérifie **seulement la présence** de la clé (sans jamais l'afficher),
puis on crée le client asynchrone.

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()  # lit .env à la racine du projet
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError("ANTHROPIC_API_KEY est vide : colle ta clé dans .env, puis relance le noyau.")

client = anthropic.AsyncAnthropic()  # lit la clé dans l'environnement
MODEL = "claude-opus-5-5"
print("Client prêt, modèle :", MODEL)

## 2. Le rôle de l'assistant

Le *system prompt* décrit le studio. On n'y met que des faits vrais : l'assistant doit dire
qu'il ne sait pas plutôt que d'inventer (prix, délais, clients).

In [ ]:
SYSTEM = """Tu es l'assistant d'ATA suarl, studio de technologies créatives à Dakar.
Services, dans cet ordre :
1. Animation 3D : films, séquences et génériques animés.
2. Expériences XR : réalité virtuelle, augmentée et installations immersives.
3. Jeux vidéo : jeux de marque, jeux sérieux, prototypes jouables.
4. IA générative : production d'images et de vidéos, outils créatifs, ateliers.
Clients visés : marques et institutions culturelles.
Réponds en français, en 3 phrases maximum. N'invente ni prix, ni délais, ni clients :
si l'information manque, dis-le et propose un échange avec l'équipe."""

## 3. Une fonction d'appel commune

`ask()` envoie une liste de messages et renvoie le texte de la réponse.
- `fallbacks="default"` : si les filtres de sécurité refusent une demande, l'API la relance
  sur un modèle de secours (option recommandée, activée ici par défaut).
- On vérifie `stop_reason` avant de lire la réponse.

In [ ]:
async def ask(messages: list[dict]) -> str:
    try:
        response = await client.beta.messages.create(
            model=MODEL,
            max_tokens=2000,
            system=SYSTEM,
            messages=messages,
            output_config={"effort": "low"},
            betas=["server-side-fallback-2026-07-01"],
            fallbacks="default",
        )
    except anthropic.AuthenticationError:
        return "[Clé API invalide : vérifie .env]"
    except anthropic.RateLimitError:
        return "[Limite de débit atteinte : réessaie dans une minute]"
    except anthropic.APIConnectionError:
        return "[Pas de connexion réseau]"
    if response.stop_reason == "refusal":
        return "[Demande refusée par les filtres de sécurité]"
    return "".join(b.text for b in response.content if b.type == "text")

## 4. Partie 1 — agent **sans mémoire**

Chaque question est envoyée **seule**. La question de suivi (« le deuxième ») n'a donc aucun
contexte : l'agent ne peut pas savoir de quoi on parle.

In [ ]:
class StatelessAgent:
    async def send(self, question: str) -> str:
        return await ask([{"role": "user", "content": question}])  # rien n'est conservé

agent = StatelessAgent()
q1 = "Quels services propose ATA suarl ?"
q2 = "Et le deuxième, qu'est-ce que ça apporterait à un musée ?"
print("Q1 :", q1, "\n→", await agent.send(q1), "\n")
print("Q2 :", q2, "\n→", await agent.send(q2))

**À observer :** la réponse à Q2 demande une précision (« le deuxième quoi ? ») ou répond
au hasard, parce que l'agent n'a jamais vu Q1.

## 5. Partie 2 — agent **avec mémoire de session**

L'API est sans état : pour « se souvenir », on renvoie **tout l'historique** à chaque tour.
La classe garde la liste `history` et y ajoute chaque question et chaque réponse.

In [ ]:
class SessionAgent:
    def __init__(self):
        self.history: list[dict] = []

    async def send(self, question: str) -> str:
        self.history.append({"role": "user", "content": question})
        answer = await ask(self.history)
        self.history.append({"role": "assistant", "content": answer})
        return answer

agent = SessionAgent()
print("Q1 :", q1, "\n→", await agent.send(q1), "\n")
print("Q2 :", q2, "\n→", await agent.send(q2))

**À observer :** cette fois, Q2 parle bien des **expériences XR** (le 2ᵉ service de la liste).

## 6. Ce que contient la mémoire

On affiche la session sous forme de tableau (pandas) : c'est exactement ce qui est renvoyé
au modèle au tour suivant. Plus la conversation est longue, plus chaque tour coûte de tokens.

In [ ]:
import pandas as pd

pd.DataFrame(
    [{"tour": i // 2 + 1, "rôle": m["role"], "extrait": m["content"][:90]} for i, m in enumerate(agent.history)]
)